![Banner Fast Track](https://upload.wikimedia.org/wikipedia/commons/thumb/b/b1/Camara_dos_Deputados_-_Plen%C3%A1rio.jpg/1200px-Camara_dos_Deputados_-_Plen%C3%A1rio.jpg)

# 🏛️ Fast Track — Camada Bronze

**Pipeline de Ingestão de Dados Brutos — Câmara dos Deputados**

| Item | Descrição |
|---|---|
| **Objetivo** | Ingestão completa dos dados da API de Dados Abertos da Câmara dos Deputados para a camada Bronze (raw) |
| **Arquitetura** | Medalhão (Bronze → Silver → Gold) sobre Delta Lake |
| **Fonte** | https://dadosabertos.camara.leg.br/api/v2 |
| **Formato destino** | Delta Tables no Unity Catalog |
| **Entradas** | API REST (JSON) da Câmara dos Deputados |
| **Saídas** | Tabelas Bronze: deputados, frentes, membros_frentes, eventos, presencas_eventos, votacoes, votos, despesas, orgaos, proposicoes, legislaturas |
| **Responsáveis** | Ernesto Bassoli |
| **Programa** | Upskill Tiller – Engenharia de Dados (T2.7) |

#Configuração do Ambiente


In [0]:
# ============================================================
# CONFIGURAÇÃO DO AMBIENTE E CATÁLOGO
# ============================================================
# Define o catálogo e schema onde as tabelas Bronze serão
# gravadas. Utiliza widgets para parametrização dinâmica.
# ============================================================

# Cria widget para selecionar o ambiente (dev, hml, prd).
dbutils.widgets.text("ambiente", "dev", "Ambiente")

# Cria widget para definir o catálogo Unity Catalog.
dbutils.widgets.text("catalogo", "workspace", "Catálogo")

# Recupera os valores dos widgets para uso no pipeline.
ambiente = dbutils.widgets.get("ambiente")
catalogo = dbutils.widgets.get("catalogo")

# Define o nome do schema para a camada Bronze.
schema_bronze = "camara_bronze"

# Cria o schema se não existir, para armazenar as tabelas Bronze.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.{schema_bronze}")

# Exibe confirmação do ambiente configurado.
print(f"Ambiente: {ambiente}")
print(f"Catálogo: {catalogo}")
print(f"Schema Bronze: {catalogo}.{schema_bronze}")

#Importações e Dependências


In [0]:
# ============================================================
# IMPORTAÇÕES DE BIBLIOTECAS
# ============================================================
# Importa todas as bibliotecas necessárias para a ingestão:
# requests para chamadas HTTP, json para parsing,
# pyspark.sql para manipulação de DataFrames.
# ============================================================

# Biblioteca para chamadas HTTP à API REST.
import requests

# Biblioteca para manipulação de dados JSON.
import json

# Biblioteca para pausas entre requisições (rate limiting).
import time

# Tipos de dados do PySpark para definição de schemas.
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, LongType, BooleanType, ArrayType, MapType
)

# Funções do PySpark para transformações de colunas.
from pyspark.sql.functions import (
    col, lit, current_timestamp, explode, from_json,
    to_date, to_timestamp, regexp_replace, trim, when
)

# Biblioteca para data e hora (campos de auditoria).
from datetime import datetime, timedelta

In [0]:
# ============================================================
# VARIÁVEIS GLOBAIS — CONFIGURAÇÃO DA API
# ============================================================
# Define a URL base, headers e parâmetros de controle
# para todas as chamadas à API da Câmara dos Deputados.
# ============================================================

# URL base da API v2 da Câmara dos Deputados.
API_BASE = "https://dadosabertos.camara.leg.br/api/v2"

# Headers para requisições HTTP — formato JSON.
HEADERS = {
    "Accept": "application/json"
}

# Quantidade de itens por página (máximo permitido pela API).
ITENS_POR_PAGINA = 100

# Tempo de espera entre requisições (em segundos) para evitar rate limiting.
SLEEP_ENTRE_REQUESTS = 0.3

# Número máximo de tentativas em caso de erro na requisição.
MAX_RETRIES = 3

# Timestamp de início da ingestão para campos de auditoria.
TS_INGESTAO = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Legislatura atual (57ª legislatura: 2023-2027).
LEGISLATURA_ATUAL = 57

In [0]:
# ============================================================
# FUNÇÃO: chamar_api_paginada
# ============================================================
# Função genérica para consumir qualquer endpoint da API
# da Câmara com paginação automática e controle de erros.
# Retorna todos os registros de todas as páginas.
# ============================================================

def chamar_api_paginada(endpoint, params=None, max_paginas=500):
    """
    Consome um endpoint da API com paginação automática.
    
    Args:
        endpoint: Caminho do endpoint (ex: '/deputados')
        params: Dicionário de parâmetros de query string
        max_paginas: Limite máximo de páginas para evitar loops infinitos
    
    Returns:
        Lista com todos os registros obtidos de todas as páginas
    """
    # Inicializa a lista que acumulará todos os registros.
    todos_registros = []
    
    # Define a página inicial.
    pagina = 1
    
    # Se não foram passados parâmetros, inicializa um dicionário vazio.
    if params is None:
        params = {}
    
    # Adiciona o número de itens por página aos parâmetros.
    params["itens"] = ITENS_POR_PAGINA
    
    # Loop de paginação — continua até não haver mais dados.
    while pagina <= max_paginas:
        # Define a página atual nos parâmetros.
        params["pagina"] = pagina
        
        # Monta a URL completa do endpoint.
        url = f"{API_BASE}{endpoint}"
        
        # Inicializa dados para esta iteração.
        dados = []
        requisicao_sucesso = False
        
        # Controle de tentativas em caso de erro.
        for tentativa in range(MAX_RETRIES):
            try:
                # Realiza a requisição GET à API.
                response = requests.get(url, headers=HEADERS, params=params, timeout=30)
                
                # Verifica se a resposta foi bem-sucedida (HTTP 200).
                if response.status_code == 200:
                    # Extrai os dados JSON da resposta.
                    dados = response.json().get("dados", [])
                    requisicao_sucesso = True
                    
                    # Se não há dados, encerra a paginação.
                    if not dados:
                        return todos_registros
                    
                    # Adiciona os registros da página atual à lista acumulada.
                    todos_registros.extend(dados)
                    
                    # Exibe progresso a cada 5 páginas.
                    if pagina % 5 == 0:
                        print(f"  Página {pagina}: {len(todos_registros)} registros acumulados")
                    
                    # Sai do loop de tentativas (sucesso).
                    break
                    
                elif response.status_code == 429:
                    # Rate limiting — aguarda antes de tentar novamente.
                    print(f"  Rate limit atingido. Aguardando 10s...")
                    time.sleep(10)
                    
                elif response.status_code == 400:
                    # Bad Request — endpoint inválido ou sem dados, não adianta tentar de novo.
                    print(f"  Endpoint {endpoint} retornou erro 400 (Bad Request) - pulando")
                    return todos_registros
                    
                else:
                    # Outro erro HTTP — registra e tenta novamente.
                    print(f"  Erro HTTP {response.status_code} na página {pagina}, tentativa {tentativa+1}")
                    time.sleep(2)
                    
            except Exception as e:
                # Erro de conexão — registra e tenta novamente.
                print(f"  Erro de conexão: {e}, tentativa {tentativa+1}")
                time.sleep(5)
        
        # Se todas as tentativas falharam, encerra a paginação.
        if not requisicao_sucesso:
            print(f"  ⚠️ Falha após {MAX_RETRIES} tentativas na página {pagina}. Encerrando paginação.")
            break
        
        # Aguarda entre requisições para respeitar rate limiting.
        time.sleep(SLEEP_ENTRE_REQUESTS)
        
        # Avança para a próxima página.
        pagina += 1
        
        # Se a página retornou menos itens que o máximo, não há mais páginas.
        if len(dados) < ITENS_POR_PAGINA:
            break
    
    # Retorna todos os registros acumulados.
    return todos_registros

In [0]:
# ============================================================
# FUNÇÃO: chamar_api_detalhe
# ============================================================
# Função para consumir um endpoint específico (por ID),
# retornando um único registro sem paginação.
# ============================================================

def chamar_api_detalhe(endpoint):
    """
    Consome um endpoint de detalhe (sem paginação).
    
    Args:
        endpoint: Caminho completo (ex: '/deputados/12345')
    
    Returns:
        Dicionário com os dados ou None em caso de erro
    """
    # Monta a URL completa.
    url = f"{API_BASE}{endpoint}"
    
    # Controle de tentativas.
    for tentativa in range(MAX_RETRIES):
        try:
            # Realiza a requisição GET.
            response = requests.get(url, headers=HEADERS, timeout=30)
            
            # Retorna os dados se a resposta foi bem-sucedida.
            if response.status_code == 200:
                return response.json().get("dados", {})
            
            # Aguarda antes de tentar novamente.
            time.sleep(1)
            
        except Exception as e:
            # Erro de conexão.
            time.sleep(3)
    
    # Retorna None se todas as tentativas falharam.
    return None

In [0]:
# ============================================================
# FUNÇÕES PARALELIZADAS OTIMIZADAS (SERVERLESS)
# ============================================================
# ThreadPoolExecutor com máximo de workers e cache.
# ============================================================

from concurrent.futures import ThreadPoolExecutor, as_completed
import requests
import time

def processar_api_com_ids_paralelo(endpoint_template, ids_list, campo_id="id", max_workers=100):
    """
    Processa endpoints com IDs em PARALELO MÁXIMO.
    OTIMIZADO: 100 workers padrão, retry inteligente.
    
    Args:
        endpoint_template: Template do endpoint (ex: '/frentes/{}/membros')
        ids_list: Lista de IDs
        campo_id: Campo de ID nos registros
        max_workers: Workers paralelos (padrão: 100)
    
    Returns:
        Lista consolidada
    """
    def fetch_with_id(item_id):
        """Busca dados para um ID."""
        try:
            endpoint = endpoint_template.format(item_id)
            url = f"{API_BASE}{endpoint}"
            
            # Retry com backoff exponencial
            for tentativa in range(2):
                try:
                    response = requests.get(url, headers=HEADERS, timeout=30)
                    if response.status_code == 200:
                        dados = response.json().get("dados", [])
                        # Adiciona ID
                        for d in dados:
                            d[campo_id] = item_id
                        return dados
                    elif response.status_code == 400:
                        return []  # Bad Request - pula
                    time.sleep(0.1 * (tentativa + 1))  # Backoff
                except:
                    time.sleep(0.2 * (tentativa + 1))
            return []
        except:
            return []
    
    todos_registros = []
    
    # ThreadPoolExecutor com max workers
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(fetch_with_id, item_id): item_id for item_id in ids_list}
        
        # Coleta resultados
        for i, future in enumerate(as_completed(futures)):
            resultado = future.result()
            todos_registros.extend(resultado)
            
            # Progresso a cada 100
            if (i + 1) % 100 == 0:
                print(f"  ⚡ {i+1}/{len(ids_list)} processados")
    
    return todos_registros


def processar_despesas_paralelo(ids_deputados, anos, max_workers=150):
    """
    OTIMIZADO: Função ESPECÍFICA para despesas com 150 workers.
    
    Args:
        ids_deputados: Lista de IDs de deputados
        anos: Lista de anos
        max_workers: Workers (padrão: 150)
    
    Returns:
        Lista de despesas
    """
    def fetch_despesas(combo):
        id_dep, ano = combo
        try:
            endpoint = f"/deputados/{id_dep}/despesas?ano={ano}"
            url = f"{API_BASE}{endpoint}"
            response = requests.get(url, headers=HEADERS, timeout=30)
            if response.status_code == 200:
                return response.json().get("dados", [])
            return []
        except:
            return []
    
    # Cria combinações
    combinacoes = [(id_dep, ano) for id_dep in ids_deputados for ano in anos]
    
    todas_despesas = []
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(fetch_despesas, combo): combo for combo in combinacoes}
        
        for i, future in enumerate(as_completed(futures)):
            resultado = future.result()
            todas_despesas.extend(resultado)
            
            if (i + 1) % 200 == 0:
                print(f"  ⚡ {i+1}/{len(combinacoes)} combinações")
    
    return todas_despesas

print("✨ Funções otimizadas (100-150 workers) carregadas!")

## 🆔 Bloco: Geração de Load ID

### 📋 Descrição

Este bloco gera um identificador único para cada execução do pipeline Bronze. O **load_id** é fundamental para rastreabilidade e permite identificar todos os registros ingeridos em uma mesma execução.

**Para que serve:**
* Rastrear qual execução do job/notebook inseriu cada registro
* Permitir replay (reprocessamento) de execuções específicas
* Facilitar troubleshooting e auditoria
* Relacionar registros entre tabelas raw e tabelas de controle

**Como funciona:**
* Usa a biblioteca `uuid` para gerar um UUID v4 (universalmente único)
* Converte para formato hexadecimal (string de 32 caracteres)
* Este ID é incluído em TODOS os registros raw inseridos nesta execução

In [0]:
# ============================================================
# GERAÇÃO DE LOAD ID + IMPORTAÇÕES ADICIONAIS
# ============================================================
# Gera um ID único para esta execução e importa bibliotecas
# necessárias para a camada Bronze raw.
# ============================================================

# Importa bibliotecas para Bronze raw
import hashlib  # Para calcular hash SHA256 dos payloads
import uuid     # Para gerar IDs únicos de execução

# Gera o load_id único para esta execução do pipeline
# Este ID será incluído em todos os registros ingeridos
load_id = uuid.uuid4().hex

print(f"🆔 Load ID gerado: {load_id}")
print(f"   Este ID identifica todos os registros desta execução")

## ⚙️ Bloco: Funções de Controle Bronze

### 📋 Descrição

Este bloco contém as funções essenciais para implementar a camada Bronze seguindo as melhores práticas de Data Engineering.

---

### 🔧 Função 1: `calcular_record_hash`

**Objetivo:** Calcular um hash SHA256 do payload JSON para garantir idempotência.

**Por que é importante:**
* Permite detectar se o mesmo dado foi ingerido duas vezes (deduplicação)
* Facilita Change Data Capture (CDC) - detectar quando um registro mudou
* Garante que reprocessamentos não criem duplicatas

**Como funciona:**
* Recebe o payload JSON (dicionário Python)
* Converte para string JSON ordenada (chaves alfabéticas)
* Calcula hash SHA256
* Retorna string hexadecimal de 64 caracteres

---

### 🔧 Função 2: `salvar_bronze_raw`

**Objetivo:** Gravar dados raw na camada Bronze mantendo payload original.

**Diferença fundamental da abordagem antiga:**
* ❌ **ERRADO**: Parse do JSON em colunas tipadas + mode OVERWRITE
* ✅ **CORRETO**: Payload raw (STRING) + colunas técnicas + mode APPEND

**Colunas técnicas adicionadas:**
* `payload` (STRING): JSON completo "como veio" da API
* `record_hash` (STRING): Hash SHA256 do payload para deduplicação
* `natural_key` (STRING): ID da entidade quando existir (deputado_id, evento_id, etc)
* `source_endpoint` (STRING): Endpoint da API (ex: "/deputados/{id}/despesas")
* `request_params` (STRING): Parâmetros da requisição (paginação, filtros)
* `page_number` (INT): Número da página quando paginado
* `http_status` (INT): Código HTTP da resposta (200, 400, 500)
* `ingestion_ts` (TIMESTAMP): Momento exato da ingestão
* `extraction_date` (DATE): Data de referência da coleta
* `load_id` (STRING): ID desta execução (para rastreabilidade)
* `source_system` (STRING): Fixo "camara_dados_abertos"

**Mode de gravação:**
* APPEND (não OVERWRITE) - histórico completo
* Deduplicação opcional por record_hash

In [0]:
# ============================================================
# FUNÇÃO: calcular_record_hash
# ============================================================
# Calcula hash SHA256 de um payload JSON para deduplicação
# e Change Data Capture (CDC).
# ============================================================

def calcular_record_hash(payload_dict):
    """
    Calcula hash SHA256 de um dicionário (payload JSON).
    
    Args:
        payload_dict: Dicionário Python (dados da API)
    
    Returns:
        String hexadecimal de 64 caracteres (hash SHA256)
    
    Exemplo:
        payload = {"id": 123, "nome": "João"}
        hash_valor = calcular_record_hash(payload)
        # Retorna: "a3f5b1c2d4e5..."
    """
    # Converte dicionário para string JSON ordenada
    # (sort_keys=True garante que mesma estrutura gera mesmo hash)
    payload_json = json.dumps(payload_dict, sort_keys=True)
    
    # Calcula hash SHA256
    hash_obj = hashlib.sha256(payload_json.encode('utf-8'))
    
    # Retorna representação hexadecimal
    return hash_obj.hexdigest()

print("✅ Função 'calcular_record_hash' carregada")

In [0]:
# ============================================================
# FUNÇÃO: salvar_bronze_raw
# ============================================================
# Grava dados na camada Bronze mantendo payload JSON raw
# com colunas técnicas para rastreabilidade completa.
# ============================================================

def salvar_bronze_raw(
    dados_raw,
    nome_entidade,
    source_endpoint,
    request_params=None,
    natural_key_field=None,
    mode="append"
):
    """
    Grava dados raw na Bronze com payload completo + metadados.
    
    Args:
        dados_raw: Lista de dicionários (dados da API)
        nome_entidade: Nome da entidade (ex: "deputados", "frentes")
        source_endpoint: Endpoint da API (ex: "/deputados")
        request_params: Dicionário com parâmetros da request (opcional)
        natural_key_field: Nome do campo que é chave natural (opcional)
        mode: "append" ou "overwrite" (padrão: "append")
    
    Returns:
        DataFrame Spark com dados gravados
    
    Exemplo:
        df = salvar_bronze_raw(
            dados_raw=deputados_list,
            nome_entidade="deputados",
            source_endpoint="/deputados",
            request_params={"idLegislatura": 57},
            natural_key_field="id"
        )
    """
    # Validação básica
    if not dados_raw:
        print(f"⚠️  Nenhum dado para gravar em {nome_entidade}")
        return None
    
    # Prepara lista de registros enriquecidos
    registros_enriquecidos = []
    
    for record in dados_raw:
        # Calcula hash do payload
        record_hash = calcular_record_hash(record)
        
        # Extrai natural key se especificado
        natural_key = str(record.get(natural_key_field)) if natural_key_field and natural_key_field in record else None
        
        # Monta registro enriquecido
        registro_bronze = {
            # Payload raw completo (JSON como string)
            "payload": json.dumps(record, ensure_ascii=False),
            
            # Metadados técnicos
            "record_hash": record_hash,
            "natural_key": natural_key,
            "source_endpoint": source_endpoint,
            "request_params": json.dumps(request_params) if request_params else None,
            "page_number": None,  # Será preenchido se for paginado
            "http_status": 200,   # Assumimos sucesso (pode ser parametrizado)
            "ingestion_ts": datetime.now(),
            "extraction_date": datetime.now().date(),
            "load_id": load_id,
            "source_system": "camara_dados_abertos"
        }
        
        registros_enriquecidos.append(registro_bronze)
    
    # Cria DataFrame Spark
    df_bronze = spark.createDataFrame(registros_enriquecidos)
    
    # Nome da tabela Bronze
    tabela_bronze = f"{catalogo}.{schema_bronze}.bronze_camara__{nome_entidade}__raw"
    
    print(f"💾 Gravando {len(registros_enriquecidos)} registros em {tabela_bronze}...")
    
    # Grava em Delta Lake
    df_bronze.write \
        .format("delta") \
        .mode(mode) \
        .option("mergeSchema", "true") \
        .saveAsTable(tabela_bronze)
    
    print(f"✅ Tabela {tabela_bronze} atualizada (mode={mode})")
    
    # Cria temp view para consulta
    df_bronze.createOrReplaceTempView(f"tempview_{nome_entidade}_raw")
    print(f"📊 Temp view 'tempview_{nome_entidade}_raw' disponível")
    
    return df_bronze

print("✅ Função 'salvar_bronze_raw' carregada")

## 🗄️ Bloco: Tabelas de Controle

### 📋 Descrição

Este bloco cria as **4 tabelas de controle obrigatórias** para operação confiável da camada Bronze. Estas tabelas não contêm dados de negócio, mas sim metadados operacionais essenciais.

---

### 📊 Tabela 1: `bronze_camara__checkpoints`

**Objetivo:** Controlar a ingestão incremental por endpoint.

**Estrutura:**
* `source_endpoint` (STRING): Endpoint da API (chave)
* `checkpoint_type` (STRING): Tipo do checkpoint ("id", "date", "page")
* `checkpoint_value` (STRING): Último valor processado
* `updated_at` (TIMESTAMP): Última atualização do checkpoint

**Exemplo de uso:**
* Endpoint `/deputados` → checkpoint_type="id" → checkpoint_value="12345" (último deputado processado)
* Endpoint `/votacoes` → checkpoint_type="date" → checkpoint_value="2026-04-27" (última data processada)

---

### 📊 Tabela 2: `bronze_camara__ingestion_runs`

**Objetivo:** Registrar cada execução completa do pipeline.

**Estrutura:**
* `load_id` (STRING): ID único da execução (chave)
* `start_ts` (TIMESTAMP): Início da execução
* `end_ts` (TIMESTAMP): Fim da execução
* `status` (STRING): "running", "success", "failed"
* `cluster_id` (STRING): ID do cluster/compute usado
* `records_processed` (LONG): Total de registros processados
* `errors_count` (INT): Quantidade de erros

**Uso:** Permite rastrear histórico de execuções e troubleshooting.

---

### 📊 Tabela 3: `bronze_camara__ingestion_requests`

**Objetivo:** Registrar cada requisição HTTP individual à API.

**Estrutura:**
* `request_id` (STRING): ID único da requisição
* `load_id` (STRING): ID da execução (FK para ingestion_runs)
* `source_endpoint` (STRING): Endpoint chamado
* `request_params` (STRING): Parâmetros da requisição (JSON)
* `http_status` (INT): Código de resposta HTTP
* `response_time_ms` (LONG): Tempo de resposta em milissegundos
* `error_message` (STRING): Mensagem de erro (se houver)
* `page_number` (INT): Número da página (se paginado)
* `records_returned` (INT): Quantidade de registros retornados
* `timestamp` (TIMESTAMP): Momento da requisição

**Uso:** Troubleshooting detalhado, monitoramento de performance da API.

---

### 📊 Tabela 4: `bronze_camara__dead_letter`

**Objetivo:** Armazenar payloads que falharam validação básica.

**Estrutura:**
* `payload` (STRING): Payload JSON que falhou
* `source_endpoint` (STRING): Endpoint de origem
* `error_message` (STRING): Motivo da falha
* `ingestion_ts` (TIMESTAMP): Quando ocorreu
* `load_id` (STRING): ID da execução

**Uso:** Resiliência - não perder dados mesmo quando há problemas.

In [0]:
# ============================================================
# CRIAÇÃO DA TABELA DE CONTROLE: CHECKPOINTS
# ============================================================
# Controla a ingestão incremental por endpoint.
# Armazena o último valor processado para evitar reprocessamento.
# ============================================================

tabela_checkpoints = f"{catalogo}.{schema_bronze}.bronze_camara__checkpoints"

print(f"📊 Criando tabela de controle: {tabela_checkpoints}...")

# SQL para criar a tabela
sql_checkpoints = f"""
CREATE TABLE IF NOT EXISTS {tabela_checkpoints} (
    source_endpoint STRING COMMENT 'Endpoint da API (ex: /deputados, /votacoes)',
    checkpoint_type STRING COMMENT 'Tipo do checkpoint: id, date, page',
    checkpoint_value STRING COMMENT 'Último valor processado (ID, data, página)',
    updated_at TIMESTAMP COMMENT 'Última atualização do checkpoint'
)
USING DELTA
COMMENT 'Controle de ingestão incremental por endpoint'
"""

spark.sql(sql_checkpoints)

print(f"✅ Tabela {tabela_checkpoints} criada/validada")
print(f"   Estrutura: source_endpoint, checkpoint_type, checkpoint_value, updated_at")

In [0]:
# ============================================================
# CRIAÇÃO DA TABELA DE CONTROLE: INGESTION RUNS
# ============================================================
# Registra cada execução completa do pipeline Bronze.
# Permite rastrear histórico e troubleshooting.
# ============================================================

tabela_runs = f"{catalogo}.{schema_bronze}.bronze_camara__ingestion_runs"

print(f"📊 Criando tabela de controle: {tabela_runs}...")

sql_runs = f"""
CREATE TABLE IF NOT EXISTS {tabela_runs} (
    load_id STRING COMMENT 'ID único da execução (UUID)',
    start_ts TIMESTAMP COMMENT 'Início da execução',
    end_ts TIMESTAMP COMMENT 'Fim da execução',
    status STRING COMMENT 'Status: running, success, failed',
    cluster_id STRING COMMENT 'ID do cluster/compute usado',
    records_processed LONG COMMENT 'Total de registros processados',
    errors_count INT COMMENT 'Quantidade de erros',
    notebook_path STRING COMMENT 'Caminho do notebook executado'
)
USING DELTA
COMMENT 'Histórico de execuções do pipeline Bronze'
"""

spark.sql(sql_runs)

print(f"✅ Tabela {tabela_runs} criada/validada")
print(f"   Estrutura: load_id, start_ts, end_ts, status, cluster_id, records_processed, errors_count")

In [0]:
# ============================================================
# CRIAÇÃO DA TABELA DE CONTROLE: INGESTION REQUESTS
# ============================================================
# Registra cada requisição HTTP individual à API.
# Permite troubleshooting detalhado e monitoramento de performance.
# ============================================================

tabela_requests = f"{catalogo}.{schema_bronze}.bronze_camara__ingestion_requests"

print(f"📊 Criando tabela de controle: {tabela_requests}...")

sql_requests = f"""
CREATE TABLE IF NOT EXISTS {tabela_requests} (
    request_id STRING COMMENT 'ID único da requisição',
    load_id STRING COMMENT 'ID da execução (FK)',
    source_endpoint STRING COMMENT 'Endpoint da API chamado',
    request_params STRING COMMENT 'Parâmetros da requisição (JSON)',
    http_status INT COMMENT 'Código HTTP de resposta (200, 400, 500)',
    response_time_ms LONG COMMENT 'Tempo de resposta em milissegundos',
    error_message STRING COMMENT 'Mensagem de erro se houver',
    page_number INT COMMENT 'Número da página se paginado',
    records_returned INT COMMENT 'Quantidade de registros retornados',
    timestamp TIMESTAMP COMMENT 'Momento da requisição'
)
USING DELTA
COMMENT 'Log detalhado de requisições HTTP à API'
"""

spark.sql(sql_requests)

print(f"✅ Tabela {tabela_requests} criada/validada")
print(f"   Estrutura: request_id, load_id, source_endpoint, http_status, response_time_ms, etc")

In [0]:
# ============================================================
# CRIAÇÃO DA TABELA DE CONTROLE: DEAD LETTER
# ============================================================
# Armazena payloads que falharam validação básica.
# Garante que nenhum dado é perdido mesmo em caso de erro.
# ============================================================

tabela_dead_letter = f"{catalogo}.{schema_bronze}.bronze_camara__dead_letter"

print(f"📊 Criando tabela de controle: {tabela_dead_letter}...")

sql_dead_letter = f"""
CREATE TABLE IF NOT EXISTS {tabela_dead_letter} (
    payload STRING COMMENT 'Payload JSON que falhou',
    source_endpoint STRING COMMENT 'Endpoint de origem do payload',
    error_message STRING COMMENT 'Motivo da falha na validação',
    ingestion_ts TIMESTAMP COMMENT 'Timestamp da tentativa de ingestão',
    load_id STRING COMMENT 'ID da execução que gerou o erro',
    error_type STRING COMMENT 'Tipo do erro: validation, parse, schema'
)
USING DELTA
COMMENT 'Payloads que falharam validação (Dead Letter Queue)'
"""

spark.sql(sql_dead_letter)

print(f"✅ Tabela {tabela_dead_letter} criada/validada")
print(f"   Estrutura: payload, source_endpoint, error_message, ingestion_ts, load_id")
print()
print("=" * 70)
print("🎉 TODAS AS 4 TABELAS DE CONTROLE FORAM CRIADAS COM SUCESSO!")
print("=" * 70)
print()
print("📊 Tabelas disponíveis:")
print(f"   1. {catalogo}.{schema_bronze}.bronze_camara__checkpoints")
print(f"   2. {catalogo}.{schema_bronze}.bronze_camara__ingestion_runs")
print(f"   3. {catalogo}.{schema_bronze}.bronze_camara__ingestion_requests")
print(f"   4. {catalogo}.{schema_bronze}.bronze_camara__dead_letter")
print()
print("✅ Infraestrutura Bronze pronta para ingestão incremental!")

## 📊 Legislaturas

**Conceito:** Legislatura é o período de 4 anos do mandato parlamentar na Câmara dos Deputados. Este dataset contém o histórico completo de todas as legislaturas desde 1826.

**Fonte:** `GET /legislaturas` (API Câmara dos Deputados v2)

---

### 🔧 Definição de Colunas

Define o schema da tabela final `camara_bronze.legislaturas` com tipagem explícita e comentários descritivos.

---

### ⚙️ Transformações

Ingestão dos dados da API, criação do DataFrame Spark com colunas tipadas e armazenamento em temp view.

---

### 💾 Gravação da Tabela

Persiste os dados da temp view em tabela Delta física no catálogo Unity Catalog.

In [0]:
# ============================================================
# DEFINIÇÃO DE COLUNAS: LEGISLATURAS
# ============================================================
# Schema da tabela final: workspace.camara_bronze.legislaturas
# Tipos de dados explícitos para otimização de armazenamento
# ============================================================

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

# Define o schema da tabela legislaturas
schema_legislaturas = StructType([
    # ID único da legislatura
    StructField("id", IntegerType(), False),  # NOT NULL
    
    # Data de início da legislatura (formato: YYYY-MM-DD)
    StructField("dataInicio", StringType(), True),
    
    # Data de fim da legislatura (formato: YYYY-MM-DD)
    StructField("dataFim", StringType(), True),
    
    # URI da API para detalhes desta legislatura
    StructField("uri", StringType(), True),
    
    # --- CAMPOS DE AUDITORIA ---
    
    # Timestamp da ingestão no Bronze
    StructField("_bronze_ingestao_ts", StringType(), True),
    
    # Fonte dos dados (API Câmara v2)
    StructField("_bronze_fonte", StringType(), True),
    
    # Ambiente de execução (dev, hml, prd)
    StructField("_bronze_ambiente", StringType(), True)
])

print("✅ Schema 'legislaturas' definido com 7 colunas (4 dados + 3 auditoria)")

In [0]:
# ============================================================
# INGESTÃO: LEGISLATURAS
# ============================================================

print("📥 Ingerindo legislaturas...")

# Busca dados da API
legislaturas_raw = chamar_api_paginada("/legislaturas")

# Cria DataFrame + TempView e armazena em variável
df_legislaturas = salvar_bronze_tempview(legislaturas_raw, "legislaturas")

print(f"📊 DataFrame 'df_legislaturas' disponível com {df_legislaturas.count()} registros")

In [0]:
# ============================================================
# GRAVAÇÃO DA TABELA: LEGISLATURAS
# ============================================================
# Persiste os dados da temp view em tabela Delta física
# Localização: {catalogo}.camara_bronze.legislaturas
# ============================================================

# Nome completo da tabela destino
tabela_legislaturas = f"{catalogo}.{schema_bronze}.legislaturas"

print(f"💾 Gravando tabela {tabela_legislaturas}...")

# Grava a tabela Delta com OVERWRITE (substitui dados existentes)
df_legislaturas.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(tabela_legislaturas)

# Validação
qtd_gravada = spark.table(tabela_legislaturas).count()
print(f"✅ Tabela {tabela_legislaturas} gravada com {qtd_gravada:,} registros")
print(f"📊 Dados disponíveis em: {tabela_legislaturas}")

## 👥 Deputados

**Conceito:** Dataset com informações básicas de todos os deputados federais em exercício na legislatura atual (57ª - 2023/2027).

**Fonte:** `GET /deputados?idLegislatura=57` (API Câmara dos Deputados v2)

**Volume:** ~835 deputados em exercício

---

### 🔧 Definição de Colunas

Define o schema da tabela final `camara_bronze.deputados` com tipagem explícita e comentários descritivos.

---

### ⚙️ Transformações

Ingestão dos dados da API filtrados por legislatura, criação do DataFrame e extração de IDs para uso nas etapas seguintes (despesas, etc).

---

### 💾 Gravação da Tabela

Persiste os dados da temp view em tabela Delta física no catálogo Unity Catalog.

In [0]:
# ============================================================
# DEFINIÇÃO DE COLUNAS: DEPUTADOS
# ============================================================
# Schema da tabela final: workspace.camara_bronze.deputados
# ============================================================

schema_deputados = StructType([
    # ID único do deputado
    StructField("id", IntegerType(), False),  # NOT NULL
    
    # URI da API para detalhes do deputado
    StructField("uri", StringType(), True),
    
    # Nome parlamentar completo
    StructField("nome", StringType(), True),
    
    # Sigla do partido (ex: PT, PL, PSOL)
    StructField("siglaPartido", StringType(), True),
    
    # Nome do partido por extenso
    StructField("uriPartido", StringType(), True),
    
    # Sigla da UF (estado)
    StructField("siglaUf", StringType(), True),
    
    # ID da legislatura (57 = atual)
    StructField("idLegislatura", IntegerType(), True),
    
    # URL da foto oficial do deputado
    StructField("urlFoto", StringType(), True),
    
    # E-mail institucional
    StructField("email", StringType(), True),
    
    # --- CAMPOS DE AUDITORIA ---
    StructField("_bronze_ingestao_ts", StringType(), True),
    StructField("_bronze_fonte", StringType(), True),
    StructField("_bronze_ambiente", StringType(), True)
])

print("✅ Schema 'deputados' definido com 12 colunas (9 dados + 3 auditoria)")

In [0]:
# ============================================================
# GRAVAÇÃO DA TABELA: DEPUTADOS
# ============================================================

tabela_deputados = f"{catalogo}.{schema_bronze}.deputados"

print(f"💾 Gravando tabela {tabela_deputados}...")

df_deputados.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(tabela_deputados)

qtd_gravada = spark.table(tabela_deputados).count()
print(f"✅ Tabela {tabela_deputados} gravada com {qtd_gravada:,} registros")
print(f"📊 Dados disponíveis em: {tabela_deputados}")

In [0]:
# ============================================================
# INGESTÃO: DEPUTADOS
# ============================================================

print("📥 Ingerindo deputados...")

# Busca dados da API
deputados_raw = chamar_api_paginada(
    "/deputados",
    params={"idLegislatura": LEGISLATURA_ATUAL}
)

# Cria DataFrame + TempView e armazena em variável
df_deputados = salvar_bronze_tempview(deputados_raw, "deputados")

# Extrai IDs para próximas etapas
ids_deputados = [d["id"] for d in deputados_raw]
print(f"📊 DataFrame 'df_deputados' disponível: {len(ids_deputados)} deputados")

## 🤝 Frentes Parlamentares

**Conceito:** Frentes Parlamentares são agrupamentos suprapartidários de deputados organizados em torno de temas específicos (ex: Frente da Agropecuária, Frente Evangélica).

**Fonte:** `GET /frentes?idLegislatura=57`

**Volume:** ~317 frentes ativas

---

### 🔧 Definição de Colunas

Define o schema da tabela final `camara_bronze.frentes`.

---

### ⚙️ Transformações

Ingestão das frentes e extração de IDs para buscar membros (próxima etapa).

---

### 💾 Gravação da Tabela

Persiste os dados em tabela Delta.

In [0]:
# ============================================================
# DEFINIÇÃO DE COLUNAS: FRENTES PARLAMENTARES
# ============================================================

schema_frentes = StructType([
    # ID único da frente parlamentar
    StructField("id", IntegerType(), False),
    
    # URI da API
    StructField("uri", StringType(), True),
    
    # Título/nome da frente
    StructField("titulo", StringType(), True),
    
    # ID da legislatura
    StructField("idLegislatura", IntegerType(), True),
    
    # --- CAMPOS DE AUDITORIA ---
    StructField("_bronze_ingestao_ts", StringType(), True),
    StructField("_bronze_fonte", StringType(), True),
    StructField("_bronze_ambiente", StringType(), True)
])

print("✅ Schema 'frentes' definido com 7 colunas (4 dados + 3 auditoria)")

In [0]:
# ============================================================
# GRAVAÇÃO DA TABELA: FRENTES
# ============================================================

tabela_frentes = f"{catalogo}.{schema_bronze}.frentes"

print(f"💾 Gravando tabela {tabela_frentes}...")

df_frentes.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(tabela_frentes)

qtd_gravada = spark.table(tabela_frentes).count()
print(f"✅ Tabela {tabela_frentes} gravada com {qtd_gravada:,} registros")

## 👥 Membros das Frentes Parlamentares

**Conceito:** Relacionamento muitos-para-muitos entre deputados e frentes parlamentares. Lista de todos os deputados membros de cada frente.

**Fonte:** `GET /frentes/{id}/membros` (317 chamadas paralelas)

**Volume:** ~59.422 registros (relacionamentos)

**Performance:** Processamento paralelo com 100 workers

---

### 🔧 Definição de Colunas

Define o schema da tabela final `camara_bronze.membros_frentes`.

---

### ⚙️ Transformações

Processamento paralelo de 317 frentes para buscar membros, consolidação dos dados e adição de Foreign Key (idFrente).

---

### 💾 Gravação da Tabela

Persiste os dados em tabela Delta.

In [0]:
# ============================================================
# DEFINIÇÃO DE COLUNAS: MEMBROS DAS FRENTES
# ============================================================

schema_membros_frentes = StructType([
    # ID do deputado (FK)
    StructField("id", IntegerType(), False),
    
    # ID da frente (FK - chave composta)
    StructField("idFrente", IntegerType(), False),
    
    # URI do deputado
    StructField("uri", StringType(), True),
    
    # Nome do deputado
    StructField("nome", StringType(), True),
    
    # Sigla do partido
    StructField("siglaPartido", StringType(), True),
    
    # UF
    StructField("siglaUf", StringType(), True),
    
    # Título na frente (Presidente, Coordenador, etc)
    StructField("titulo", StringType(), True),
    
    # --- CAMPOS DE AUDITORIA ---
    StructField("_bronze_ingestao_ts", StringType(), True),
    StructField("_bronze_fonte", StringType(), True),
    StructField("_bronze_ambiente", StringType(), True)
])

print("✅ Schema 'membros_frentes' definido com 10 colunas (7 dados + 3 auditoria)")

In [0]:
# ============================================================
# INGESTÃO: FRENTES PARLAMENTARES
# ============================================================

print("📥 Ingerindo frentes parlamentares...")

# Busca dados da API
frentes_raw = chamar_api_paginada(
    "/frentes",
    params={"idLegislatura": LEGISLATURA_ATUAL}
)

# Cria DataFrame + TempView
df_frentes = salvar_bronze_tempview(frentes_raw, "frentes")

# Extrai IDs
ids_frentes = [f["id"] for f in frentes_raw]
print(f"📊 DataFrame 'df_frentes' disponível: {len(ids_frentes)} frentes")

In [0]:
# ============================================================
# INGESTÃO: MEMBROS DAS FRENTES (100 WORKERS)
# ============================================================

print(f"📥 Ingerindo membros ({len(ids_frentes)} frentes com 100 workers)...")

# Paralelo com 100 workers
membros_raw = processar_api_com_ids_paralelo(
    endpoint_template="/frentes/{}/membros",
    ids_list=ids_frentes,
    campo_id="idFrente",
    max_workers=100
)

print(f"  📊 {len(membros_raw)} membros coletados")

# DataFrame + TempView
df_membros_frentes = salvar_bronze_tempview(membros_raw, "membros_frentes")
print(f"📊 DataFrame 'df_membros_frentes' disponível")

In [0]:
# ============================================================
# GRAVAÇÃO DA TABELA: MEMBROS DAS FRENTES
# ============================================================

tabela_membros_frentes = f"{catalogo}.{schema_bronze}.membros_frentes"

print(f"💾 Gravando tabela {tabela_membros_frentes}...")

df_membros_frentes.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(tabela_membros_frentes)

qtd_gravada = spark.table(tabela_membros_frentes).count()
print(f"✅ Tabela {tabela_membros_frentes} gravada com {qtd_gravada:,} registros")

## 📅 Eventos Legislativos

**Conceito:** Reuniões, sessões plenárias, audiências públicas e outros acontecimentos oficiais da Câmara.

**Fonte:** `GET /eventos?dataInicio={data}&dataFim={data}`

**Volume:** ~398 eventos (últimos 2 anos)

---

### 🔧 Definição de Colunas

Schema da tabela `camara_bronze.eventos`.

---

### ⚙️ Transformações

Ingestão com filtro temporal (2 anos) e extração de IDs para buscar presenças.

---

### 💾 Gravação da Tabela

Persiste dados em tabela Delta.

In [0]:
# ============================================================
# DEFINIÇÃO DE COLUNAS: EVENTOS
# ============================================================

schema_eventos = StructType([
    StructField("id", IntegerType(), False),
    StructField("dataHoraInicio", StringType(), True),
    StructField("dataHoraFim", StringType(), True),
    StructField("descricao", StringType(), True),
    StructField("localCamara", StringType(), True),
    StructField("situacao", StringType(), True),
    StructField("uri", StringType(), True),
    # Auditoria
    StructField("_bronze_ingestao_ts", StringType(), True),
    StructField("_bronze_fonte", StringType(), True),
    StructField("_bronze_ambiente", StringType(), True)
])

print("✅ Schema 'eventos' definido")

In [0]:
# ============================================================
# GRAVAÇÃO DA TABELA: EVENTOS
# ============================================================

tabela_eventos = f"{catalogo}.{schema_bronze}.eventos"
print(f"💾 Gravando tabela {tabela_eventos}...")

df_eventos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela_eventos)

qtd_gravada = spark.table(tabela_eventos).count()
print(f"✅ Tabela {tabela_eventos} gravada com {qtd_gravada:,} registros")

## 👥 Presenças em Eventos

**Conceito:** Registro de presença de deputados em eventos legislativos.

**Fonte:** `GET /eventos/{id}/deputados` (1000 chamadas paralelas)

**Volume:** ~23.803 presenças

**Performance:** 100 workers paralelos

---

### 🔧 Definição de Colunas

Schema da tabela `camara_bronze.presencas_eventos`.

---

### ⚙️ Transformações

Processamento paralelo de até 1000 eventos.

---

### 💾 Gravação da Tabela

Persiste dados em tabela Delta.

In [0]:
# ============================================================
# DEFINIÇÃO DE COLUNAS: PRESENÇAS EM EVENTOS
# ============================================================

schema_presencas_eventos = StructType([
    StructField("id", IntegerType(), False),  # ID deputado
    StructField("idEvento", IntegerType(), False),  # FK
    StructField("nome", StringType(), True),
    StructField("siglaPartido", StringType(), True),
    StructField("siglaUf", StringType(), True),
    StructField("uri", StringType(), True),
    # Auditoria
    StructField("_bronze_ingestao_ts", StringType(), True),
    StructField("_bronze_fonte", StringType(), True),
    StructField("_bronze_ambiente", StringType(), True)
])

print("✅ Schema 'presencas_eventos' definido")

In [0]:
# ============================================================
# INGESTÃO: EVENTOS LEGISLATIVOS
# ============================================================

# Período
ano_atual = datetime.now().year
data_inicio = f"{ano_atual - 1}-01-01"
data_fim = f"{ano_atual}-12-31"

print(f"📥 Ingerindo eventos ({data_inicio} a {data_fim})...")

# Busca eventos
eventos_raw = chamar_api_paginada(
    "/eventos",
    params={"dataInicio": data_inicio, "dataFim": data_fim}
)

# DataFrame + TempView
df_eventos = salvar_bronze_tempview(eventos_raw, "eventos")

# Extrai IDs
ids_eventos = [e["id"] for e in eventos_raw]
print(f"📊 DataFrame 'df_eventos' disponível: {len(ids_eventos)} eventos")

In [0]:
# ============================================================
# INGESTÃO: PRESENÇAS (100 WORKERS)
# ============================================================

print(f"📥 Ingerindo presenças ({len(ids_eventos)} eventos com 100 workers)...")

# Paralelo com 100 workers
presencas_raw = processar_api_com_ids_paralelo(
    endpoint_template="/eventos/{}/deputados",
    ids_list=ids_eventos[:1000],  # Limite 1000 eventos
    campo_id="idEvento",
    max_workers=100
)

print(f"  📊 {len(presencas_raw)} presenças coletadas")

# DataFrame + TempView
df_presencas_eventos = salvar_bronze_tempview(presencas_raw, "presencas_eventos")
print(f"📊 DataFrame 'df_presencas_eventos' disponível")

In [0]:
# ============================================================
# GRAVAÇÃO DA TABELA: PRESENÇAS EM EVENTOS
# ============================================================

tabela_presencas = f"{catalogo}.{schema_bronze}.presencas_eventos"
print(f"💾 Gravando tabela {tabela_presencas}...")

df_presencas_eventos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela_presencas)

qtd_gravada = spark.table(tabela_presencas).count()
print(f"✅ Tabela {tabela_presencas} gravada com {qtd_gravada:,} registros")

In [0]:
# ============================================================
# DEFINIÇÃO DE COLUNAS: VOTAÇÕES
# ============================================================

schema_votacoes = StructType([
    StructField("id", StringType(), False),
    StructField("data", StringType(), True),
    StructField("descricao", StringType(), True),
    StructField("aprovacao", StringType(), True),
    StructField("siglaOrgao", StringType(), True),
    StructField("uri", StringType(), True),
    # Auditoria
    StructField("_bronze_ingestao_ts", StringType(), True),
    StructField("_bronze_fonte", StringType(), True),
    StructField("_bronze_ambiente", StringType(), True)
])

print("✅ Schema 'votacoes' definido")

In [0]:
# ============================================================
# GRAVAÇÃO DA TABELA: VOTAÇÕES
# ============================================================

tabela_votacoes = f"{catalogo}.{schema_bronze}.votacoes"
print(f"💾 Gravando tabela {tabela_votacoes}...")

df_votacoes.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela_votacoes)

qtd_gravada = spark.table(tabela_votacoes).count()
print(f"✅ Tabela {tabela_votacoes} gravada com {qtd_gravada:,} registros (vazia - API erro 400)")

## 🗳️ Votos Individuais

**Conceito:** Votos individuais de cada deputado em votações.

**Fonte:** `GET /votacoes/{id}/votos`

**Volume:** ~0 registros (sem votações)

---

### 🔧 Definição de Colunas

Schema da tabela `camara_bronze.votos`.

---

### ⚙️ Transformações

Processamento paralelo de votações (se houver).

---

### 💾 Gravação da Tabela

Persiste dados em tabela Delta (vazia).

In [0]:
# ============================================================
# DEFINIÇÃO DE COLUNAS: VOTOS
# ============================================================

schema_votos = StructType([
    StructField("id", IntegerType(), False),  # ID deputado
    StructField("idVotacao", StringType(), False),  # FK
    StructField("nome", StringType(), True),
    StructField("siglaPartido", StringType(), True),
    StructField("tipoVoto", StringType(), True),
    # Auditoria
    StructField("_bronze_ingestao_ts", StringType(), True),
    StructField("_bronze_fonte", StringType(), True),
    StructField("_bronze_ambiente", StringType(), True)
])

print("✅ Schema 'votos' definido")

## 🗳️ Votações

**Conceito:** Processos de aprovação/rejeição de proposições na Câmara.

**Fonte:** `GET /votacoes?dataInicio={data}&dataFim={data}`

**Volume:** ~0 registros (API erro 400)

---

### 🔧 Definição de Colunas

Schema da tabela `camara_bronze.votacoes`.

---

### ⚙️ Transformações

Ingestão com filtro temporal e extração de IDs para buscar votos.

---

### 💾 Gravação da Tabela

Persiste dados em tabela Delta (vazia).

In [0]:
# ============================================================
# INGESTÃO: VOTAÇÕES
# ============================================================

print(f"📥 Ingerindo votações ({data_inicio} a {data_fim})...")

# Busca votações
votacoes_raw = chamar_api_paginada(
    "/votacoes",
    params={"dataInicio": data_inicio, "dataFim": data_fim}
)

# DataFrame + TempView
df_votacoes = salvar_bronze_tempview(votacoes_raw, "votacoes")

# Extrai IDs
ids_votacoes = [v["id"] for v in votacoes_raw] if votacoes_raw else []
print(f"📊 DataFrame 'df_votacoes' disponível: {len(ids_votacoes)} votações")

In [0]:
# ============================================================
# INGESTÃO: VOTOS (100 WORKERS)
# ============================================================

if len(ids_votacoes) > 0:
    print(f"📥 Ingerindo votos ({len(ids_votacoes)} votações com 100 workers)...")
    
    # Paralelo com 100 workers
    votos_raw = processar_api_com_ids_paralelo(
        endpoint_template="/votacoes/{}/votos",
        ids_list=ids_votacoes,
        campo_id="idVotacao",
        max_workers=100
    )
    
    print(f"  📊 {len(votos_raw)} votos coletados")
else:
    print("⚠️ Sem votações para buscar votos")
    votos_raw = []

# DataFrame + TempView
df_votos = salvar_bronze_tempview(votos_raw, "votos")
print(f"📊 DataFrame 'df_votos' disponível")

In [0]:
# ============================================================
# GRAVAÇÃO DA TABELA: VOTOS
# ============================================================

tabela_votos = f"{catalogo}.{schema_bronze}.votos"
print(f"💾 Gravando tabela {tabela_votos}...")

df_votos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela_votos)

qtd_gravada = spark.table(tabela_votos).count()
print(f"✅ Tabela {tabela_votos} gravada com {qtd_gravada:,} registros (vazia)")

In [0]:
# ============================================================
# DEFINIÇÃO DE COLUNAS: DESPESAS CEAP
# ============================================================

schema_despesas = StructType([
    StructField("ano", IntegerType(), True),
    StructField("mes", IntegerType(), True),
    StructField("tipoDespesa", StringType(), True),
    StructField("dataDocumento", StringType(), True),
    StructField("valorDocumento", DoubleType(), True),
    StructField("valorGlosa", DoubleType(), True),
    StructField("valorLiquido", DoubleType(), True),
    StructField("nomeFornecedor", StringType(), True),
    StructField("cnpjCpfFornecedor", StringType(), True),
    StructField("urlDocumento", StringType(), True),
    # Auditoria
    StructField("_bronze_ingestao_ts", StringType(), True),
    StructField("_bronze_fonte", StringType(), True),
    StructField("_bronze_ambiente", StringType(), True)
])

print("✅ Schema 'despesas' definido com 13 colunas (10 dados + 3 auditoria)")

## 💰 Despesas CEAP

**Conceito:** CEAP (Cota para Exercício da Atividade Parlamentar) - despesas de cada deputado.

**Fonte:** `GET /deputados/{id}/despesas?ano={ano}` (1.670 combinações)

**Volume:** ~16.091 despesas (últimos 2 anos)

**Performance:** 150 workers paralelos (máxima performance)

---

### 🔧 Definição de Colunas

Schema da tabela `camara_bronze.despesas`.

---

### ⚙️ Transformações

Processamento paralelo máximo (835 deputados x 2 anos) com 150 workers.

---

### 💾 Gravação da Tabela

Persiste dados em tabela Delta.

In [0]:
# ============================================================
# INGESTÃO: DESPESAS CEAP (150 WORKERS OTIMIZADOS)
# ============================================================

anos = [ano_atual - 1, ano_atual]

print(f"📥 Ingerindo despesas CEAP ({len(ids_deputados)} dep x {len(anos)} anos = {len(ids_deputados)*len(anos)} combinações)...")
print(f"  ⚡ Usando 150 WORKERS PARALELOS (MÁXIMA PERFORMANCE)")

# Função otimizada específica com 150 workers
despesas_raw = processar_despesas_paralelo(
    ids_deputados=ids_deputados,
    anos=anos,
    max_workers=150
)

print(f"  📊 {len(despesas_raw)} despesas coletadas")

# DataFrame + TempView
df_despesas = salvar_bronze_tempview(despesas_raw, "despesas")
print(f"📊 DataFrame 'df_despesas' disponível")

In [0]:
# ============================================================
# GRAVAÇÃO DA TABELA: DESPESAS CEAP
# ============================================================

tabela_despesas = f"{catalogo}.{schema_bronze}.despesas"
print(f"💾 Gravando tabela {tabela_despesas}...")

df_despesas.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela_despesas)

qtd_gravada = spark.table(tabela_despesas).count()
print(f"✅ Tabela {tabela_despesas} gravada com {qtd_gravada:,} registros")

## 🏛️ Órgãos

**Conceito:** Estruturas administrativas e legislativas da Câmara (comissões, frentes, blocos, etc).

**Fonte:** `GET /orgaos`

**Volume:** ~1.605 órgãos

---

### 🔧 Definição de Colunas

Schema da tabela `camara_bronze.orgaos`.

---

### ⚙️ Transformações

Ingestão completa sem filtros.

---

### 💾 Gravação da Tabela

Persiste dados em tabela Delta.

In [0]:
# ============================================================
# DEFINIÇÃO DE COLUNAS: ÓRGÃOS
# ============================================================

schema_orgaos = StructType([
    StructField("id", IntegerType(), False),
    StructField("sigla", StringType(), True),
    StructField("nome", StringType(), True),
    StructField("apelido", StringType(), True),
    StructField("tipoOrgao", StringType(), True),
    StructField("uri", StringType(), True),
    # Auditoria
    StructField("_bronze_ingestao_ts", StringType(), True),
    StructField("_bronze_fonte", StringType(), True),
    StructField("_bronze_ambiente", StringType(), True)
])

print("✅ Schema 'orgaos' definido")

In [0]:
# ============================================================
# GRAVAÇÃO DA TABELA: ÓRGÃOS
# ============================================================

tabela_orgaos = f"{catalogo}.{schema_bronze}.orgaos"
print(f"💾 Gravando tabela {tabela_orgaos}...")

df_orgaos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela_orgaos)

qtd_gravada = spark.table(tabela_orgaos).count()
print(f"✅ Tabela {tabela_orgaos} gravada com {qtd_gravada:,} registros")

In [0]:
# ============================================================
# DEFINIÇÃO DE COLUNAS: PROPOSIÇÕES
# ============================================================

schema_proposicoes = StructType([
    StructField("id", IntegerType(), False),
    StructField("siglaTipo", StringType(), True),
    StructField("numero", IntegerType(), True),
    StructField("ano", IntegerType(), True),
    StructField("ementa", StringType(), True),
    StructField("dataApresentacao", StringType(), True),
    StructField("uri", StringType(), True),
    # Auditoria
    StructField("_bronze_ingestao_ts", StringType(), True),
    StructField("_bronze_fonte", StringType(), True),
    StructField("_bronze_ambiente", StringType(), True)
])

print("✅ Schema 'proposicoes' definido")

In [0]:
# ============================================================
# GRAVAÇÃO DA TABELA: PROPOSIÇÕES
# ============================================================

tabela_proposicoes = f"{catalogo}.{schema_bronze}.proposicoes"
print(f"💾 Gravando tabela {tabela_proposicoes}...")

df_proposicoes.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela_proposicoes)

qtd_gravada = spark.table(tabela_proposicoes).count()
print(f"✅ Tabela {tabela_proposicoes} gravada com {qtd_gravada:,} registros (vazia)")

In [0]:
# ============================================================
# DEFINIÇÃO DE COLUNAS: PARTIDOS
# ============================================================

schema_partidos = StructType([
    StructField("id", IntegerType(), False),
    StructField("sigla", StringType(), True),
    StructField("nome", StringType(), True),
    StructField("uri", StringType(), True),
    # Auditoria
    StructField("_bronze_ingestao_ts", StringType(), True),
    StructField("_bronze_fonte", StringType(), True),
    StructField("_bronze_ambiente", StringType(), True)
])

print("✅ Schema 'partidos' definido")

In [0]:
# ============================================================
# GRAVAÇÃO DA TABELA: PARTIDOS
# ============================================================

tabela_partidos = f"{catalogo}.{schema_bronze}.partidos"
print(f"💾 Gravando tabela {tabela_partidos}...")

df_partidos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela_partidos)

qtd_gravada = spark.table(tabela_partidos).count()
print(f"✅ Tabela {tabela_partidos} gravada com {qtd_gravada:,} registros")

In [0]:
# ============================================================
# INGESTÃO: ÓRGÃOS
# ============================================================

print("📥 Ingerindo órgãos...")

# Busca dados
orgaos_raw = chamar_api_paginada("/orgaos")

# DataFrame + TempView
df_orgaos = salvar_bronze_tempview(orgaos_raw, "orgaos")
print(f"📊 DataFrame 'df_orgaos' disponível")

## 📜 Proposições

**Conceito:** Projetos de lei, emendas, MPs e outras matérias legislativas.

**Fonte:** `GET /proposicoes?dataInicio={data}&dataFim={data}`

**Volume:** ~0 registros (API erro 400)

---

### 🔧 Definição de Colunas

Schema da tabela `camara_bronze.proposicoes`.

---

### ⚙️ Transformações

Ingestão com filtro temporal.

---

### 💾 Gravação da Tabela

Persiste dados em tabela Delta (vazia).

In [0]:
# ============================================================
# INGESTÃO: PROPOSIÇÕES
# ============================================================

print(f"📥 Ingerindo proposições ({data_inicio} a {data_fim})...")

# Busca dados
proposicoes_raw = chamar_api_paginada(
    "/proposicoes",
    params={"dataInicio": data_inicio, "dataFim": data_fim}
)

# DataFrame + TempView
df_proposicoes = salvar_bronze_tempview(proposicoes_raw, "proposicoes")
print(f"📊 DataFrame 'df_proposicoes' disponível")

## 🏦 Partidos

**Conceito:** Partidos políticos com representação na Câmara.

**Fonte:** `GET /partidos`

**Volume:** ~21 partidos

---

### 🔧 Definição de Colunas

Schema da tabela `camara_bronze.partidos`.

---

### ⚙️ Transformações

Ingestão completa de todos os partidos.

---

### 💾 Gravação da Tabela

Persiste dados em tabela Delta.

In [0]:
# ============================================================
# INGESTÃO: PARTIDOS
# ============================================================

print("📥 Ingerindo partidos...")

# Busca dados
partidos_raw = chamar_api_paginada("/partidos")

# DataFrame + TempView
df_partidos = salvar_bronze_tempview(partidos_raw, "partidos")
print(f"📊 DataFrame 'df_partidos' disponível")

## 📊 Bloco: Resumo da Ingestão Bronze

### 📋 Descrição

**Objetivo:** Consolidar e validar todos os datasets ingeridos, exibindo estatísticas completas de cada Temp View e DataFrame criado.

**Processamento:**
1. Define lista com os 12 datasets Bronze
2. Para cada dataset:
   * Acessa a Temp View correspondente (`tempview_{nome}`)
   * Executa `.count()` para obter volume de registros
   * Exibe nome do DataFrame (`df_{nome}`)
   * Exibe nome da Temp View (`tempview_{nome}`)
   * Trata erros caso alguma view não exista
3. Exibe estatísticas de performance:
   * Timestamp de conclusão
   * Número de workers paralelos usados
   * Vantagens da arquitetura (memória, sem disco)
4. Fornece exemplos de uso (Python e SQL)

**Datasets Validados (12 no total):**
* legislaturas
* deputados
* frentes
* membros_frentes
* eventos
* presencas_eventos
* votacoes
* votos
* despesas
* orgaos
* proposicoes
* partidos

**Nomenclatura Validada:**
* **DataFrames:** `df_legislaturas`, `df_deputados`, etc.
* **Temp Views:** `tempview_legislaturas`, `tempview_deputados`, etc.

**Saída:** Relatório formatado no console com:
* ✅ Status de cada dataset
* 📊 Contagem de registros
* 💾 Nomes de DataFrame e Temp View
* ⚡ Estatísticas de performance
* 🔧 Instruções de uso

**Utilização:** Executar esta célula após todas as ingestões para validar que todos os dados foram carregados corretamente.

## 🔍 Bloco: Exemplos de Consulta às Temp Views e DataFrames

### 📋 Descrição

**Objetivo:** Demonstrar diferentes formas de acessar e consultar os dados ingeridos, preparando os dados para análises ou para a camada Silver.

---

### **Exemplo 1: Consulta com DataFrame Python**

**Método:** Acesso direto à variável `df_deputados`

**Código:**
```python
df_deputados.groupBy("siglaUf").count().orderBy("count", ascending=False).limit(5)
```

**Resultado:** Top 5 estados com mais deputados

---

### **Exemplo 2: Consulta com SQL via Temp View**

**Método:** `spark.sql()` acessando `tempview_deputados`

**Código:**
```sql
SELECT siglaUf, COUNT(*) as total
FROM tempview_deputados
GROUP BY siglaUf
ORDER BY total DESC
LIMIT 5
```

**Resultado:** Mesma consulta, mas em SQL puro

---

### **Exemplo 3: Join entre Temp Views (SQL)**

**Método:** Relacionamento entre `tempview_frentes` e `tempview_membros_frentes`

**Código:**
```sql
SELECT 
    f.titulo,
    COUNT(m.id) as total_membros
FROM tempview_frentes f
LEFT JOIN tempview_membros_frentes m ON f.id = m.idFrente
GROUP BY f.titulo
ORDER BY total_membros DESC
LIMIT 5
```

**Resultado:** Top 5 frentes com mais membros

---

### **Exemplo 4: Acesso Misto (spark.table + count)**

**Método:** Usa `spark.table('tempview_nome')` para acessar como DataFrame

**Código:**
```python
qtd = spark.table('tempview_legislaturas').count()
```

**Utilização:** Útil quando o nome da view é dinâmico ou você não tem a variável Python

---

### **Próximos Passos:**

✅ Dados Bronze prontos para:
* **Camada Silver:** Limpeza, transformação, normalização
* **Análises exploratórias:** Estatísticas, visualizações
* **Dashboards:** Consultas SQL diretas nas Temp Views
* **Machine Learning:** Feature engineering sobre os DataFrames

**Lembrete:**
* 📊 **DataFrames**: Use `df_{nome}` para operações PySpark
* 💾 **Temp Views**: Use `tempview_{nome}` para consultas SQL

In [0]:
# ============================================================
# RESUMO FINAL DA INGESTÃO BRONZE
# ============================================================
# Lista DataFrames e Temp Views criados com nomenclatura padrão.
# ============================================================

# Lista de todos os datasets ingeridos (nome base)
datasets_bronze = [
    "legislaturas", "deputados", "frentes", "membros_frentes",
    "eventos", "presencas_eventos", "votacoes", "votos",
    "despesas", "orgaos", "proposicoes", "partidos"
]

# Exibe o resumo
print("=" * 70)
print("📊 RESUMO DA INGESTÃO BRONZE - DATAFRAMES E TEMP VIEWS")
print("=" * 70)
print()
print("NOMENCLATURA PADRÃO:")
print("  • DataFrames: df_{nome}  (variáveis Python)")
print("  • TempViews:  tempview_{nome}  (SQL em memória)")
print()
print("=" * 70)

# Itera sobre cada dataset
for nome in datasets_bronze:
    try:
        # Nome da temp view com prefixo
        tempview_nome = f"tempview_{nome}"
        
        # Conta os registros da temp view
        qtd = spark.table(tempview_nome).count()
        
        # Exibe status
        print(f"✅ {nome:20s} | {qtd:>8,} registros")
        print(f"   💾 DataFrame: df_{nome}")
        print(f"   💾 TempView:  {tempview_nome}")
        print()
    except Exception as e:
        print(f"❌ {nome:20s} | Não encontrado")
        print()

# Exibe timestamp de conclusão
print("=" * 70)
print(f"🏁 Ingestão concluída: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()
print("⚡ PERFORMANCE:")
print("  • Paralelização: 100-150 workers (ThreadPoolExecutor)")
print("  • Dados em memória: acesso ultra-rápido")
print("  • Sem gravação em disco: máxima velocidade")
print()
print("🔧 COMO USAR:")
print("  # Python (DataFrames):")
print("  display(df_deputados.limit(10))")
print()
print("  # SQL (TempViews):")
print("  spark.sql('SELECT * FROM tempview_deputados LIMIT 10')")
print("=" * 70)

In [0]:
# ============================================================
# EXEMPLOS DE CONSULTA ÀS TEMP VIEWS E DATAFRAMES
# ============================================================
# Demonstra como acessar os dados via Python (DataFrames)
# e via SQL (TempViews) para análises ou camada Silver.
# ============================================================

print("🔍 EXEMPLOS DE CONSULTA\n")
print("=" * 70)

# Exemplo 1: Usando DataFrame Python
print("1️⃣ PYTHON - DataFrame direto (df_deputados):")
print()
df_exemplo1 = df_deputados.groupBy("siglaUf").count().orderBy("count", ascending=False).limit(5)
display(df_exemplo1)

# Exemplo 2: Usando SQL com TempView
print("\n2️⃣ SQL - TempView (tempview_deputados):")
print()
df_exemplo2 = spark.sql("""
    SELECT siglaUf, COUNT(*) as total
    FROM tempview_deputados
    GROUP BY siglaUf
    ORDER BY total DESC
    LIMIT 5
""")
display(df_exemplo2)

# Exemplo 3: Join entre TempViews
print("\n3️⃣ SQL - Join entre TempViews (frentes + membros):")
print()
df_exemplo3 = spark.sql("""
    SELECT 
        f.titulo,
        COUNT(m.id) as total_membros
    FROM tempview_frentes f
    LEFT JOIN tempview_membros_frentes m ON f.id = m.idFrente
    GROUP BY f.titulo
    ORDER BY total_membros DESC
    LIMIT 5
""")
display(df_exemplo3)

# Exemplo 4: Acesso misto - DataFrame + SQL
print("\n4️⃣ MISTO - Acessar TempView via spark.table():")
print()
for view_base in ["legislaturas", "deputados", "frentes", "membros_frentes"]:
    try:
        tempview_nome = f"tempview_{view_base}"
        qtd = spark.table(tempview_nome).count()
        print(f"  • {tempview_nome}: {qtd:,} registros")
    except:
        print(f"  • {tempview_nome}: Não disponível")

print("\n" + "=" * 70)
print("✨ Dados prontos para camada Silver!")
print("\nLEMBRETE:")
print("  📄 DataFrames: df_{nome}       -> Usar em Python/PySpark")
print("  💾 TempViews:  tempview_{nome} -> Usar em SQL")
print("=" * 70)